# Export Data untuk Tableau - Olist E-Commerce

Menyiapkan data dalam format denormalized (flat table) untuk dashboard Tableau.

**Prasyarat:** Jalankan `01_load_to_sqlite.ipynb` dan `03_cleaning.ipynb` terlebih dahulu.

**Output:** `data/tableau/orders_full.csv` — tabel utama (orders + customers + items + products)

## Persiapan

In [ ]:
import os
import sqlite3

import pandas as pd

In [ ]:
DB_PATH = "../data/database/olist.db"
TABLEAU_DIR = "../data/tableau/"

os.makedirs(TABLEAU_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

---
## 1. Tabel Utama (Denormalized)

Join semua tabel menjadi satu flat table dengan kolom turunan yang siap dipakai di Tableau.

In [ ]:
df = pd.read_sql_query("""
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    p.product_category_name,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_item oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
""", conn)

print(f"Baris: {len(df)}, Kolom: {len(df.columns)}")
df.head()

In [ ]:
# Konversi datetime
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [ ]:
# Kolom turunan untuk Tableau

# Waktu
df["order_date"] = df["order_purchase_timestamp"].dt.date
df["order_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["order_day_of_week"] = df["order_purchase_timestamp"].dt.day_name()
df["order_hour"] = df["order_purchase_timestamp"].dt.hour

# Pengiriman
df["delivery_days"] = (
    (df["order_delivered_customer_date"] - df["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
).round(1)

df["estimated_days"] = (
    (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
).round(1)

df["delivery_diff_days"] = (df["delivery_days"] - df["estimated_days"]).round(1)

df["is_late"] = df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]

# Finansial
df["total_value"] = (df["price"] + df["freight_value"]).round(2)

# Dimensi produk
df["product_volume_cm3"] = (
    df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"]
).round(0)

print(f"Kolom final: {len(df.columns)}")
df.dtypes

In [ ]:
# Export (separator ; dan desimal ,)
output_path = os.path.join(TABLEAU_DIR, "orders_full.csv")
df.to_csv(output_path, index=False, sep=";", decimal=",")

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Disimpan: {output_path} ({size_mb:.1f} MB, {len(df)} baris, {len(df.columns)} kolom)")

---
## Ringkasan

| File | Baris | Kolom | Gunakan Untuk |
|------|-------|-------|---------------|
| `orders_full.csv` | 108.628 | 27 | Dashboard utama: tren, produk, pengiriman, geografi |

### Kolom Turunan

| Kolom | Tipe | Deskripsi |
|-------|------|----------|
| `order_date` | Date | Tanggal order (tanpa waktu) |
| `order_year_month` | String | Bulan order (format: 2017-11) |
| `order_day_of_week` | String | Hari dalam seminggu (Monday, dll) |
| `order_hour` | Integer | Jam pembelian (0-23) |
| `delivery_days` | Float | Waktu pengiriman aktual (hari) |
| `estimated_days` | Float | Waktu estimasi (hari) |
| `delivery_diff_days` | Float | Selisih aktual - estimasi (positif = terlambat) |
| `is_late` | Boolean | Apakah terlambat (True/False) |
| `total_value` | Float | Harga + ongkir |
| `product_volume_cm3` | Float | Volume produk (panjang x tinggi x lebar) |

In [ ]:
conn.close()